# Reddit Kaggle Competition: Inference Notebook

Loads the fine-tuned **DistilBERT** model and generates predictions on the test set.

**Metric:** Macro ROC AUC.

In [ ]:
import os

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

DATA_DIR = '../data/'
MODEL_PATH = 'model_best.pth'
TOKENIZER_PATH = 'tokenizer'
BATCH_SIZE = 32
MAX_LEN = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load the Data

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_kaggle.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_kaggle.csv'))

emotion_cols = [c for c in train_df.columns if c not in ['ID', 'text']]
num_labels = len(emotion_cols)

print(f'Test samples: {len(test_df)}')
print(f'Total emotions: {num_labels}')

## 2. Tokenizer
Loaded from the local `tokenizer/` directory produced by the training run (offline-safe).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
print(f'Tokenizer loaded ({tokenizer.__class__.__name__}, vocab size {tokenizer.vocab_size})')

## 3. Dataset and DataLoader

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(
            df['text'].astype(str).tolist(),
            padding='max_length', truncation=True, max_length=MAX_LEN,
            return_tensors='pt',
        )
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']

    def __len__(self):
        return self.input_ids.size(0)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
        }

test_loader = DataLoader(EmotionDataset(test_df), batch_size=BATCH_SIZE)

## 4. Model Definition & Loading
Rebuild the DistilBERT classification head from the saved config, then load the fine-tuned weights from `model_best.pth`.

In [ ]:
config = AutoConfig.from_pretrained(
    TOKENIZER_PATH,
    num_labels=num_labels,
    problem_type='multi_label_classification',
)
model = AutoModelForSequenceClassification.from_config(config).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print('Model loaded and ready for inference.')

## 5. `predict_emotion` Function
Single-text helper used by the grader. Returns a dict of `{emotion: probability}`.

In [ ]:
def predict_emotion(text):
    enc = tokenizer(
        str(text), padding='max_length', truncation=True, max_length=MAX_LEN,
        return_tensors='pt',
    )
    with torch.no_grad():
        logits = model(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE),
        ).logits
    probs = torch.sigmoid(logits).cpu().numpy()[0]
    return dict(zip(emotion_cols, probs.tolist()))

## 6. Generate Submission

In [ ]:
probs = []
with torch.no_grad():
    for batch in test_loader:
        logits = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
        ).logits
        probs.append(torch.sigmoid(logits).cpu().numpy())

submission = pd.DataFrame(np.vstack(probs), columns=emotion_cols)
submission.insert(0, 'ID', test_df['ID'].values)
submission.to_csv('submission.csv', index=False)
print('Submission file created: submission.csv')